# CQI Data Analysis — Key Findings

Analysis of the Coffee Quality Institute dataset: what drives cup quality,
how processing methods compare, and where climate research fits in.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

from utils.data import load_cqi

df = load_cqi()

cupping_attrs = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance"]
print(f"{len(df)} coffees loaded from {df['country_of_origin'].nunique()} countries")

1311 coffees loaded from 36 countries


## 1. What is Cupping?

Cupping is the standardized method for scoring coffee quality. Each coffee is rated on six
sensory attributes (roughly 6.5–8.5 scale each):

| Attribute | What it measures |
|---|---|
| **Aroma** | How the coffee smells (dry and wet) |
| **Flavor** | Taste on the palate |
| **Aftertaste** | What lingers after swallowing |
| **Acidity** | Brightness and liveliness (high = desirable, not sour) |
| **Body** | Weight and mouthfeel |
| **Balance** | How well all attributes work together |

Three additional scores (uniformity, clean cup, sweetness) are nearly always perfect 10s
in this dataset, so they don't differentiate coffees.

**Total Cup Points** is the sum of all attributes. In this dataset it ranges from 78 to 89.

In [2]:
# Score ranges for each attribute
stats = df[cupping_attrs + ["total_cup_points"]].describe().T[["min", "mean", "max"]]
stats.columns = ["Min", "Mean", "Max"]
stats = stats.round(2)

fig = go.Figure(data=go.Table(
    header=dict(values=["Attribute"] + list(stats.columns)),
    cells=dict(values=[
        [c.replace("_", " ").title() for c in stats.index],
        stats["Min"], stats["Mean"], stats["Max"]
    ]),
))
fig.update_layout(title="Cupping Score Ranges", height=350)
fig.show()

## 2. What Drives Total Cup Score?

**Flavor and aftertaste are the strongest predictors of total score** (r=0.94).
Body matters least (r=0.85). A coffee that tastes great and finishes clean
will score high even if it's lighter-bodied.

In [3]:
# Correlation of each attribute with total cup points
corrs = df[cupping_attrs + ["total_cup_points"]].corr()["total_cup_points"].drop("total_cup_points")
corrs = corrs.sort_values(ascending=True)

fig = px.bar(
    x=corrs.values,
    y=[c.replace("_", " ").title() for c in corrs.index],
    orientation="h",
    labels={"x": "Correlation (r)", "y": "Attribute"},
    title="Correlation of Each Attribute with Total Cup Score",
    text=[f"{v:.3f}" for v in corrs.values],
)
fig.update_layout(yaxis=dict(categoryorder="total ascending"), showlegend=False)
fig.show()

## 3. Cupping Attributes Are Highly Correlated

Good coffee tends to score well across all dimensions (r=0.63–0.88 between attributes).
**Body is the most independent attribute** — a coffee can have great aroma but average body.
This means body is potentially the most interesting dimension for differentiating
processing methods.

In [4]:
# Attribute inter-correlation heatmap
corr_matrix = df[cupping_attrs].corr()

fig = px.imshow(
    corr_matrix,
    x=[c.replace("_", " ").title() for c in cupping_attrs],
    y=[c.replace("_", " ").title() for c in cupping_attrs],
    color_continuous_scale="YlOrBr",
    text_auto=".2f",
    title="Cupping Attribute Inter-Correlation",
    labels={"color": "r"},
)
fig.show()

## 4. Processing Methods and Fermentation

Processing method determines how the coffee cherry is handled after picking,
which directly controls fermentation:

| Method | Fermentation | Typical Flavor Profile |
|---|---|---|
| **Washed** (124 samples) | Fruit removed before drying — minimal fermentation | Clean, bright, high clarity |
| **Natural/Dry** (46 samples) | Dried with fruit intact — extended fermentation | Fruity, winey, heavier body |
| **Honey/Pulped Natural** (25 samples) | Partial fruit removal — moderate fermentation | In between — sweet, balanced |
| **Experimental** (anaerobic, carbonic, etc.) | Controlled fermentation environments | Intense, complex (very few samples) |

**Key finding: processing method doesn't dramatically affect total score.**
Washed (83.6), Natural (83.7), and Honey (83.6) average nearly the same.
The differences appear in *which* attributes score higher.

In [5]:
# Average score by processing method
main_methods = ["Washed / Wet", "Natural / Dry", "Pulped natural / honey"]
method_df = df[df["processing_method"].isin(main_methods)]

method_avg = method_df.groupby("processing_method")[cupping_attrs].mean()

fig = go.Figure()
for method in main_methods:
    vals = method_avg.loc[method]
    fig.add_trace(go.Scatterpolar(
        r=vals.values.tolist() + [vals.values[0]],
        theta=[c.replace("_", " ").title() for c in cupping_attrs] + [cupping_attrs[0].replace("_", " ").title()],
        name=method,
        fill="toself",
        opacity=0.5,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[7.3, 8.0])),
    title="Cupping Profile by Processing Method (Radar)",
)
fig.show()

In [6]:
# Score distributions by processing method
melted = method_df.melt(
    id_vars=["processing_method"],
    value_vars=cupping_attrs,
    var_name="attribute",
    value_name="score",
)
melted["attribute"] = melted["attribute"].str.replace("_", " ").str.title()

fig = px.box(
    melted,
    x="attribute",
    y="score",
    color="processing_method",
    title="Score Distributions: Where Processing Methods Differ",
    labels={"attribute": "", "score": "Score", "processing_method": "Processing"},
)
fig.update_layout(boxmode="group")
fig.show()

## 5. Altitude Has a Weak Effect

Higher altitude is often cited as producing better coffee (cooler temps = slower cherry
maturation = more complex sugars). In this dataset, the correlation is **weak (r=0.17)**.

This could be due to:
- Small sample size (207 coffees)
- Messy altitude data (ranges like "1700-1930", missing values)
- Altitude alone doesn't capture the full climate picture — temperature, humidity,
  and rainfall matter too. This is where NASA POWER data comes in.

In [ ]:
# Altitude vs score, colored by processing method
alt_df = df.dropna(subset=["altitude_mean_meters", "total_cup_points"])
alt_df = alt_df[alt_df["altitude_mean_meters"] < 5000]

fig = px.scatter(
    alt_df,
    x="altitude_mean_meters",
    y="total_cup_points",
    color="processing_method",
    trendline="ols",
    hover_data=["farm_name", "country_of_origin", "region"],
    labels={"altitude_mean_meters": "Altitude (m)", "total_cup_points": "Total Cup Points"},
    title=f"Altitude vs Cup Score — {len(alt_df)} samples (r=0.11)",
    opacity=0.7,
)
fig.show()

## 6. Country Profiles

Different origins have distinct scoring patterns, shaped by terroir, common varieties,
and preferred processing methods.

In [8]:
# Top countries by average score
top_countries = df["country_of_origin"].value_counts().head(8).index
country_df = df[df["country_of_origin"].isin(top_countries)]

fig = px.box(
    country_df,
    x="country_of_origin",
    y="total_cup_points",
    color="processing_method",
    hover_data=["farm_name", "variety"],
    labels={"country_of_origin": "", "total_cup_points": "Total Cup Points"},
    title="Score Distribution by Country and Processing Method",
)
fig.update_layout(xaxis=dict(categoryorder="total descending"))
fig.show()

## 7. Next Steps — Climate + Fermentation

The key insight from this analysis: **processing method shifts the flavor profile
more than the total score**. To understand fermentation's role more deeply, we need
to layer in climate data:

- **Temperature during harvest** — affects fermentation speed and microbial activity
- **Humidity** — influences drying rate for naturals and honeys
- **Rainfall patterns** — determines harvest timing and cherry ripeness

The `nasa_power.py` scraper can pull this data for any farm coordinate.
The question to answer: **do certain climates favor certain processing methods,
and does that show up in the cupping scores?**